In [2]:
import sys
import os
from pathlib import Path
import argparse
print("Creating Spark Session...")


PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "spark"))

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from common.config_loader import load_config
from framework.metadata.entity_config_loader import (load_entity_config)
from framework.spark.spark_session import create_spark_session
from framework.iceberg.schema_loader import load_platform_schema
from framework.iceberg.table_manager import (
    ensure_namespace_exists,
    create_table_if_not_exists,
    write_to_iceberg
)
from framework.logging.logger import get_logger



Creating Spark Session...


In [4]:


    
    entity_name = 'customer'

    #Read environment variable for ENV, default to "local" if not set
    env = os.getenv("ENV", "local")
    print(f"Environment: {env}")

    print(f"Starting Bronze Stream for Entity: {entity_name}")

    

    #Load configuration for environment and entity
    env_config = load_config(env)
    entity_config = load_entity_config(entity_name)
    

    #logging configs
    logger = get_logger(
    f"bronze_Stream/{entity_name}",
    env_config
    )

    ##Read configuration values for Kafka, root folder for storage paths and checkpoint paths from the loaded configurations
    bootstrap_servers = env_config["kafka"]["bootstrap_servers"]
    topic = entity_config["source"]["topic"]
    bronze_root = env_config["storage"]["bronze_root_path"]
    checkpoint_path = f"{env_config['storage']['checkpoint_root_path']}/bronze/{entity_config['entity_name']}"
    #Bronze schema path
    bronze_schema_path = PROJECT_ROOT / "configs" / "framework" / "bronze_schema.yaml"
    

    #create the bronze path using the entity name from the entity configuration
    bronze_path = (
        f"{bronze_root}/"
        f"{entity_config['entity_name']}"
    )
    
    

   

Environment: local
Starting Bronze Stream for Entity: customer


In [8]:
 #create Spark Session
spark = create_spark_session("Bronze Stream", env)

print("Spark Session Created")

   

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/14 19:32:10 WARN Utils: Your hostname, Sauravs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.8 instead (on interface en0)
26/07/14 19:32:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/sauravpandey/Projects/streaming/subscription-platform/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/sauravpandey/.ivy2.5.2/cache
The jars for the packages stored in: /Users/sauravpandey/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
com.google.cloud.bigdataoss#gcs-connector added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9e6b1e92-2e48-485a-ab54-7191e2b2258f;1.0
	confs: [default]
	foun

Spark Session Created


In [9]:
bronze_schema = load_platform_schema(bronze_schema_path)
catalog = env_config["iceberg"]["catalog_name"]
bronze_table = f"{catalog}.bronze.{entity_name}"
ensure_namespace_exists(
        spark,
        catalog,
        "bronze",
        logger

    )
create_table_if_not_exists(
        spark,
        catalog,
        "bronze",
        entity_name,
        bronze_schema,
        logger

    )

    

 

2026-07-14 19:33:26,961 | INFO     | bronze_Stream/customer | Columns - business: ['key BINARY', 'value BINARY NOT NULL', 'topic STRING NOT NULL', 'partition INT NOT NULL', 'offset BIGINT NOT NULL', 'timestamp TIMESTAMP', 'timestampType INT NOT NULL']
2026-07-14 19:33:26,962 | INFO     | bronze_Stream/customer | Columns - Full: ['key BINARY', 'value BINARY NOT NULL', 'topic STRING NOT NULL', 'partition INT NOT NULL', 'offset BIGINT NOT NULL', 'timestamp TIMESTAMP', 'timestampType INT NOT NULL']


In [10]:
   #Read data from Kafka topic and write to Bronze layer in Parquet format
df = (
        spark.readStream
        .format("kafka")
        .option("failOnDataLoss", False)
        .option("kafka.bootstrap.servers", bootstrap_servers)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load()
    )
print("Kafka Stream Created")

df.printSchema()



Kafka Stream Created
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [11]:
    #write to Iceberg Bronze table
query = (df.writeStream.foreachBatch(
        lambda batch_df, batch_id: write_to_iceberg(
            batch_df,
            batch_id,
            bronze_table,
            mode="append",
            logger= logger
        )
    ).option("checkpointLocation", checkpoint_path)

    .start()
    )

print("Bronze Streaming Query Started")



26/07/14 19:34:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Bronze Streaming Query Started


In [15]:
spark.sql("SELECT * FROM insightflow.bronze.customer")

DataFrame[key: binary, value: binary, topic: string, partition: int, offset: bigint, timestamp: timestamp, timestampType: int]

In [17]:
bronze_df = (
        spark.read
        .table(bronze_table)
    )
print("2. Bronze Stream Created\n",bronze_df.show(20, truncate=False))

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------